# Basic Pinch and `dt_cont` Sensitivity
This notebook starts from the single-case `PinchProblem` front door, then uses `PinchWorkspace` for a named `dt_cont` sensitivity study on the packaged crude preheat train example.


## Case context
`crude_preheat_train.json` is a compact refinery preheat-train case with enough structure to show direct targets, graph interpretation, and how a small change in the minimum temperature approach assumption moves the utility picture. Notebook 4 picks up the real multiperiod derivative of this same case.


In [ ]:
import pandas as pd
import plotly.express as px

from IPython.display import display

from OpenPinch import PinchProblem, PinchWorkspace


In [ ]:
problem = PinchProblem("crude_preheat_train.json", project_name="crude_preheat_train")
validated = problem.validate()
problem.target()
summary = problem.summary_frame()

{
    "stream_count": len(validated.streams),
    "utility_count": len(validated.utilities or []),
    "period_ids": list(problem.period_ids),
}


In [ ]:
summary


## Read the standard direct-integration graphs
The single-case API already exposes the graph catalog and the common figure accessors. For this process-only example, the direct-integration row is the main screen to interpret.


In [ ]:
catalog = problem.plot.catalog()
catalog.loc[
    catalog["Target"] == "crude_preheat_train/Direct Integration",
    ["Target", "Graph Type", "Graph Name"],
].reset_index(drop=True)


In [ ]:
display(problem.plot.composite_curve(zone_name="crude_preheat_train"))
display(problem.plot.shifted_composite_curve(zone_name="crude_preheat_train"))
display(problem.plot.grand_composite_curve(zone_name="crude_preheat_train"))


## Appendix: `area_cost()` return fields
The public target accessor also exposes the area-and-cost route. This cell reports the returned fields on the same real case so you can see what the current package build returns without leaving the main workflow.


In [ ]:
area_cost_target = problem.target.area_cost()
pd.DataFrame(
    [
        {
            "Target": area_cost_target.name,
            "Area": area_cost_target.area,
            "Number of Units": area_cost_target.num_units,
            "Capital Cost": area_cost_target.capital_cost,
            "Total Cost": area_cost_target.total_cost,
        }
    ]
)


## Named `dt_cont` sensitivity with `PinchWorkspace`
Once the single-case picture is clear, `PinchWorkspace` becomes the right tool for named baseline-versus-variant studies. Here the baseline stays intact while copied cases sweep a wide range of active `dt_cont` assumptions through `set_dt_cont_multiplier(...)`.


In [ ]:
workspace = PinchWorkspace(
    source="crude_preheat_train.json",
    project_name="crude_preheat_train",
)

target_name = "crude_preheat_train/Direct Integration"
dt_cont_multipliers = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.5, 10.0]

case_rows = []
for multiplier in dt_cont_multipliers:
    case_name = "baseline" if multiplier == 1.0 else f"dt_cont_x{str(multiplier).replace('.', '_')}"
    if case_name != "baseline":
        workspace.copy_case("baseline", case_name, activate=False)
        workspace.set_dt_cont_multiplier(multiplier, case_name=case_name)

    detailed_summary = workspace.summary_frame(case_name=case_name, detailed=True)
    site_row = detailed_summary.loc[detailed_summary["Target"] == target_name].iloc[0]
    case_rows.append(
        {
            "case": case_name,
            "dt_cont_multiplier": multiplier,
            "Hot Utility Target / kW": site_row["Qh (value)"],
            "Cold Utility Target / kW": site_row["Qc (value)"],
            "Heat Recovery / kW": site_row["Qr (value)"],
            "Hot Pinch / degC": site_row["Hot Pinch (value)"],
            "Cold Pinch / degC": site_row["Cold Pinch (value)"],
        }
    )

dt_cont_results = pd.DataFrame(case_rows).sort_values("dt_cont_multiplier").reset_index(drop=True)
dt_cont_results


In [ ]:
selected_dt_cases = ["dt_cont_x0_25", "dt_cont_x0_5", "dt_cont_x2_0", "dt_cont_x4_0"]

pd.concat(
    [
        workspace.compare_cases("baseline", case_name, target_name=target_name)
        .loc[["Change"]]
        .assign(case=case_name)
        for case_name in selected_dt_cases
    ],
    axis=0,
).reset_index(drop=True)


## Plot the target response
Plotting the targets across the multiplier range makes the tradeoff visible: tighter approach temperatures reduce external utility demand, while wider approach temperatures increase hot and cold utility requirements and reduce recoverable heat.


In [ ]:
dt_cont_plot_data = dt_cont_results.melt(
    id_vars=["dt_cont_multiplier"],
    value_vars=[
        "Hot Utility Target / kW",
        "Cold Utility Target / kW",
        "Heat Recovery / kW",
    ],
    var_name="Target",
    value_name="Heat Flow / kW",
)

fig = px.line(
    dt_cont_plot_data,
    x="dt_cont_multiplier",
    y="Heat Flow / kW",
    color="Target",
    markers=True,
    title="Direct-integration targets across dt_cont multipliers",
)
fig.update_layout(
    xaxis_title="dt_cont multiplier",
    yaxis_title="Heat Flow / kW",
    legend_title_text="Target",
)
fig


## Next step
This case is scalar, so the direct results above stay in one operating period. Notebook 4 switches to `crude_preheat_train_multiperiod.json` and shows the same direct targeting surface on real named periods such as `turndown`, `base`, and `peak`.
